# Fourier Neural Operators for Parametric PDEs

A single layer of the neural operator is defined as:

$$\mathcal{L} := \sigma(\mathcal{W} + \mathcal{K} + b)$$

$$G_\theta := \mathcal{Q} \circ \mathcal{L} \circ \mathcal{P}$$

where:
- $\mathcal{P} : \mathbb{R}^{d_a} \rightarrow \mathbb{R}^{d_v}$ is a lifting layer
- $\mathcal{Q} : \mathbb{R}^{d_v} \rightarrow \mathbb{R}^{d_u}$ is a projection layer
- $\mathcal{L} : \mathbb{R}^{d_v} \rightarrow \mathbb{R}^{d_v}$ is the Neural
  Operator Layer with:
  - $\mathcal{K}$ is the Fourier kernel operator
  - $\mathcal{W}$ is a local linear operator (a "skip connection" matrix)
  - $b$ is a "function" bias

## Fourier Layer

The Fourier Neural Operator uses:

$$\mathcal{K}(v) = \mathcal{F}^{-1}(R_\phi \cdot \mathcal{F}(v))$$

where $\mathcal{F}(v)$ are the Fourier coefficients of the input
function $v$, and $R_\phi$ is the learnable kernel in frequency space.

Here $d_a$ represents the input channels, $d_v$ the hidden channels, and
$d_u$ the output channels.

In [ ]:
# Imports

import equinox as eqx
import jax
import jax.numpy as jnp
import numpy as np
import optax
from fno_models import FNO1D
from jaxtyping import Array, Float
from tqdm import tqdm
from utils import load_and_preprocess_data, ProgressPlotter, visualize_results

Define the loss function. In this case the model maps from initial condition 
to the rest of the solution

In [ ]:
def loss_fn(
    model: FNO1D,
    batch: Float[Array, "B T W C"],
) -> Array:
    # Predict using the FNO model
    predictions = jax.vmap(model)(batch[:, 0:1])

    # Compute mean squared error loss
    mse_loss = jnp.mean((predictions - batch[:, 1:]) ** 2)

    return mse_loss


@eqx.filter_jit
def training_step(
    model: FNO1D,
    optimizer,
    opt_state,
    batch: Array,
):
    """Single training step with gradient computation and parameter update."""

    @eqx.filter_value_and_grad
    def compute_loss(model):
        return loss_fn(model, batch)

    loss_value, grads = compute_loss(model)
    updates, opt_state = optimizer.update(grads, opt_state, model)
    model = eqx.apply_updates(model, updates)

    return model, opt_state, loss_value


def train_fno_model(
    model: FNO1D,
    train_dataloader,
    test_dataloader=None,
    n_epochs: int = 100,
    batch_size: int = 64,
    learning_rate: float = 5e-4,
    grad_clip_norm: float = 1.0,
    visualize_every_n_epochs: int = 1,
    save_plots: bool = True,
):
    print("Training FNO model...")

    # Initialize progress plotter
    plotter = None
    if save_plots and test_dataloader is not None:
        plotter = ProgressPlotter(
            output_dir="tmp_fno",
            model_name="FNO",
            framerate=10,
        )

    # Calculate total training steps for proper scheduling
    batches_per_epoch = len(train_dataloader)
    total_steps = n_epochs * batches_per_epoch

    # Setup optimizer with cosine schedule and gradient clipping
    schedule = optax.cosine_onecycle_schedule(
        transition_steps=total_steps,
        peak_value=learning_rate,
    )
    optimizer = optax.chain(
        optax.clip_by_global_norm(grad_clip_norm),
        optax.adamw(schedule),
    )
    opt_state = optimizer.init(eqx.filter(model, eqx.is_array))

    # Pre-allocate losses array for nice plotting
    losses = np.full(n_epochs, np.nan)  # Initialize with NaN
    losses_list = []  # Keep track of actual losses for returning

    # Training loop
    with tqdm(range(n_epochs), desc="Training FNO") as pbar:
        for epoch in pbar:
            epoch_losses = []

            # Iterate through batches from dataloader
            for batch in train_dataloader:
                model, opt_state, loss_value = training_step(
                    model,
                    optimizer,
                    opt_state,
                    batch,
                )
                epoch_losses.append(loss_value)

            # Record average loss for this epoch
            avg_loss = jnp.mean(jnp.array(epoch_losses))
            losses_list.append(avg_loss)
            losses[epoch] = avg_loss  # Simple numpy array assignment
            pbar.set_postfix({"Loss": f"{avg_loss:.6f}"})

            # Visualize results periodically
            if plotter is not None and (epoch + 1) % visualize_every_n_epochs == 0:
                plotter(
                    model,
                    test_dataloader,
                    losses,
                    show_plot=False,
                    show_loss_plot=False,
                )

    return model, np.array(losses_list), plotter

Setup training parameters and data

In [ ]:
hidden_channels = 32
n_modes = 101
n_layers = 4
n_steps_train = 100  # Number of time steps in training data
n_steps_test = 200  # Number of time steps in test data

n_epochs = 1000
batch_size = 128
learning_rate = 5e-3
grad_clip_norm = 1.0

In [ ]:

(
    train_dataloader,
    val_dataloader,
    test_dataloader,
) = load_and_preprocess_data(
    "data/string_nonlin_100_Gaussian_16000Hz_1.0s.npy",
    batch_size=batch_size,
    n_steps_train=n_steps_train,
    n_steps_test=n_steps_test,
)

# Get a sample batch to determine data shapes
sample = next(iter(train_dataloader))
print(f"Sample input shape: {sample[0].shape}")
print(f"Sample target shape: {sample[1].shape}")
# Setup FNO model parameters
n_spatial_points = sample.shape[2]  # spatial dimension
input_channels = sample.shape[-1]  # 1 * channels
output_channels = sample.shape[3]
n_prediction_steps = sample.shape[1] - 1  # Use actual target sequence length

# Create FNO model
key = jax.random.PRNGKey(42)
fno_model = FNO1D(
    input_channels=input_channels,
    hidden_channels=hidden_channels,
    n_modes=n_modes,
    output_channels=output_channels,
    n_layers=n_layers,
    n_steps=n_prediction_steps,
    key=key,
)

Train the model

In [ ]:
# Train the model
trained_fno, training_losses, plotter = train_fno_model(
    fno_model,
    train_dataloader,
    test_dataloader=test_dataloader,
    n_epochs=n_epochs,
    batch_size=batch_size,
    learning_rate=learning_rate,
    grad_clip_norm=grad_clip_norm,
    save_plots=False,
)

In [ ]:
visualize_results(
    trained_fno,
    test_dataloader,
    training_losses,
    model_name="FNO",
    show_loss_plot=False,
)

# Generate animation from training frames
if plotter is not None:
    plotter.render_animation("fno_training.webm")

The model is able to learn the mapping from initial condition to the full solution.
- How well does it generalize to different initial conditions?
- How could we adapt this to learn to map from escitation to full 
solution, for example? What are some drawbacks of this approach?
- How would we include physical parameters as inputs to the model?